# 學生學習與作業版｜任務 2：檢查資料品質與日期

**姓名：** 王翊綺　**學號／組別：** C114118117　**完成日期：** 11/11

**學習目標：** 辨認缺失、重複、異常與含歧義的日期，保留原始紀錄。

本單元與教師版「任務 2」相同標題、資料、教學順序與範例；除了明確標示的作業與填空，其餘教師版教材內容保留。對照重點：程式填空對應教師版「2.2 異常值」。

### 如何完成
1. 先閱讀觀念與執行資料準備；各單元可以獨立啟動，不需要依賴上一單元的變數。
2. **💻 P 作業**：在指定程式 cell 補上 `____`，可先用提供的 Vibe Coding 提示詞詢問 AI。補完再執行，並執行後面的驗證 cell。未填時出現 `NameError` 是預期現象，先完成填空再繼續。
3. **📝 Q 作業**：查詢 AI、比較教材內容，填寫緊接的文字作答區；請記錄實際提示詞、自己的解釋與查證方式。
4. **🎯 M 小任務**：在任務 4、8、12、16 將前面幾單元整合成可執行的成果。不要只交 AI 回覆，要交實際程式、輸出與自己的解釋。
5. 把 Notebook 存到自己的 Google 雲端硬碟，完成後下載／依教師指示繳交 `.ipynb`；小任務保留輸出、文字與 AI 使用紀錄。

**案例限制：**本資料是公式生成的 60 筆模擬紀錄，不能推論真實學生或因果。缺失與 15 小時值在計分後加入。預測時欄位可得性是假設；全部資料已在前半段探索，後續切出的測試集僅示範流程，不能視為完全獨立的新資料。

**完成檢核：**本單元有 1 項程式填空、2 項 AI 查詢作答；若是任務 4、8、12、16，另外有 1 項整合小任務。請保留自己的輸出、回答和驗證紀錄。


## 共用資料準備
以下在記憶體重建與既有教材相同的資料。執行後可以直接開始本單元；不會覆寫 CSV。


In [1]:
# 匯入套件：numpy 建立隨機資料，pandas 管理表格，pyplot 繪圖
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 固定種子，並維持呼叫順序，以重現原教材的 60 筆模擬資料
rng = np.random.default_rng(42)
n = 60
departments = ["資訊管理", "企業管理", "金融管理"]
# 每個字典鍵是一個欄位；對應的值是 60 名學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)
# 成績依設定公式與隨機誤差建立；關係來自模擬規則，不能視為因果發現
noise = rng.normal(0, 5, n)
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)
# 故意在計分後加入資料品質問題；無法從表格還原計分時原始值
df.loc[[5, 17], "sleep_hours"] = np.nan
df.loc[8, "screen_time_hours"] = 15.0
print("模擬資料形狀（列, 欄）：", df.shape)


模擬資料形狀（列, 欄）： (60, 7)


## 2.1 缺失與重複
**觀念說明：**缺失表示未知，不能當成 0。重複整列與識別碼重複是兩件事；先確認業務意義再決定是否刪除。
**預測：**睡眠欄位有幾筆缺失？其他欄位如何？


In [2]:
# isna() 找缺失；sum() 在每欄加總 True 的數量
print(df.isna().sum())
print("整列完全重複：", df.duplicated().sum())
print("學生編號重複：", df["student_id"].duplicated().sum())
# 顯示含睡眠缺失的原始列，先觀察而不是直接刪除
display(df.loc[df["sleep_hours"].isna()])


student_id           0
department           0
sleep_hours          2
screen_time_hours    0
study_hours          0
attendance_rate      0
quiz_score           0
dtype: int64
整列完全重複： 0
學生編號重複： 0


,student_id,department,sleep_hours,screen_time_hours,study_hours,attendance_rate,quiz_score
5,6,金融管理,NaN,2.8,4.3,96.0,92.0
17,18,資訊管理,NaN,4.5,2.9,100.0,82.0


## 📝 學生作業 02-Q1｜AI 查詢與作答：未知與缺失
**任務：**「未知」、0、NaN 有什麼差異？未確認日期為何不能直接猜？

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請比較未知、數值零、缺失值，並解釋 01-02-2026 在日月格式不明時應怎麼處理。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q1 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：kklc；我是 Python 與資料分析初學者。請比較未知、數值零、缺失值，並解釋 01-02-2026 在日月格式不明時應怎麼處理。請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：先用「問卷：你這星期花了幾小時讀書？」當例子：
| 類型 | 學生實際情況 | 資料欄位裡應該長什麼樣 |
|---|---|---|
| **數值零 (0)** | 有回答，確實 0 小時（沒讀書） | `0` |
| **缺失值 (missing / NA)** | 完全沒回答、系統沒抓到 | `NaN`（pandas 預設的缺失值） |
| **未知 (unknown)** | 有回答，但內容是「我不確定 / 拒答」 | 文字 `unknown`，或獨立分類 |

核心觀念：三者意義不同，處理方式也不能一樣。

零是真實的測量結果，計算平均時要算進去。
缺失值是「沒有資料」，pandas 的 .mean() 預設會跳過 NaN。
未知是一個「有效答案」，代表受訪者給了一個答案，只是內容為未知。它不該被當成 0，也不該直接混進數值平均。

- 我的理解與具體例子：數值0是真正意義上的0；NaN是沒回答或系統沒抓到；未知是可能對方拒答或不知道。(例子如下)
  01-02-2026 日月格式不明怎麼辦？
01-02-2026 可能是 1 月 2 日（月-日，美國慣例）或 2 月 1 日（日-月，台灣、歐洲慣例）。單看一行資料無解，但可以分四步處理：

步驟 1：先找「外部規則」
查資料字典（data dictionary）、問卷說明、或資料來源的地區慣例。如果資料是台灣單位做的，通常是 日-月-年（DD-MM-YYYY）。

步驟 2：用「整份資料」自己驗證格式
關鍵技巧： 看有沒有出現 13 以上的數字。月份最多只有 12，所以：

import pandas as pd

s = pd.Series(["01-02-2026", "15-03-2026", "25-12-2025"])

first = s.str.split("-").str[0].astype(int)   # 第一組數字

second = s.str.split("-").str[1].astype(int)  # 第二組數字

print("第一組數字最大：", first.max())   # 15 → 超過 12，第一組不可能是月份

print("第二組數字最大：", second.max())  # 12

第一組出現 15（> 12）→ 證明 第一組是日，格式確定是 DD-MM-YYYY，所以 01-02-2026 = 2026 年 2 月 1 日。
反之，如果第二組出現 > 12，格式就是 MM-DD-YYYY。
如果兩組數字全都 ≤ 12，資料本身無法自證，就要回到步驟 1 找外部規則，或向資料提供者確認。
步驟 3：用「明確格式」轉換，不要靠自動猜測

#確認是 DD-MM-YYYY 後，明確指定 format 轉換

d = pd.to_datetime(s, format="%d-%m-%Y")

print(d)

為什麼要明確寫 format？ 如果不寫，pandas 會自動猜測，遇到 01-02-2026 這種兩邊都合法的日期，不同版本、不同環境可能猜出不同結果，而且速度比較慢。寫明 %d-%m-%Y 或 %m-%d-%Y，結果才穩定、可重現。

步驟 4：把「假設」記錄下來
如果你最終是憑慣例（例如「台灣資料採 DD-MM」）決定的，就在程式碼旁加註解，例如：

#假設 DD-MM-YYYY：資料來源為 XX 單位問卷，未附資料字典，依台灣慣例處理

這樣以後（或別人）重新分析時才知道你是怎麼決定的。

- 查證來源／方式及日期（無法查證請註明）：查證日期： 2026-11-11（台北時間，UTC+08:00）

  | 來源 | 查證方式 | 狀態 |
|---|---|---|
| pandas 官方 API 文件 `pandas.to_datetime`（文件版號 3.1.1）| 網頁全文抓取 | 成功 |
| ISO 官方頁面「ISO 8601 — Date and time format」| 直接抓取被防火牆擋下（雲端驗證頁），改以該站的搜尋摘要取得內容 | 部分成功，引用時已標明 |
| 本地 Python 環境實際執行 | pandas 3.0.2 實跑 | 成功 |

- 我檢查或修正 AI 回答的地方：無


## 2.2 異常值需要依規格判斷
15 小時是在產生成績之後故意置入，超過本案例原始設定的 10 小時上限；真實生活中不能僅憑超過 10 小時就認定錯誤。先標記、查證與記錄。


## 💻 學生作業 02-P1｜Vibe Coding 程式填空
**對應教師版：**任務 2「2.2 異常值」的示範程式。

**任務：**建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。

**執行方式：**找到下一個程式儲存格中的 `____`，使用自己撰寫或 AI 建議的程式補齊，保留其他已提供的程式與註解。補完後先執行該 cell，再執行緊接的自動檢查；後續單元內容可能依賴此變數。

**可使用的 Vibe Coding 提示詞：**
```text
我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。
我的任務是：建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
```
完成後記錄實際使用的提示詞與自己改動的地方。


In [3]:
# 根據本案例原先生成規格標記高於 10 小時的紀錄
flag = df["screen_time_hours"] > 10  # TODO：找出本案例超過 10 小時的資料
print("超過案例生成上限的筆數：", flag.sum())
display(df.loc[flag, ["student_id", "screen_time_hours", "quiz_score"]])


超過案例生成上限的筆數： 1


,student_id,screen_time_hours,quiz_score
8,9,15.0,80.0


### ✅ 作業 02-P1｜執行驗證
執行下方程式，並同時檢查上方實際輸出。自動檢查只涵蓋主要條件；如果圖表、說明或資料來源不符任務，仍需人工修正。


In [4]:
assert flag.equals(df["screen_time_hours"].gt(10))
assert len(flag) == len(df)
print("檢查通過：篩選條件正確。")


檢查通過：篩選條件正確。


### ✍️ 作業 02-P1｜我的完成紀錄
- 我實際使用的提示詞（若未使用 AI 請註明）：我是 Python 與資料分析初學者，正在 Google Colab 使用 pandas、numpy、matplotlib 或 scikit-learn。我的任務是：建立布林篩選條件 flag，判斷 screen_time_hours 是否大於 10；請勿刪除或修改資料。
請依我貼上的上下文，只協助補上 ____，保留指定變數名稱與後續程式，不修改原始 df。
請解釋每一步的作用，提供繁體中文註解及可手動驗證的方法。
不要虛構執行結果；如需額外資訊，先指出缺少什麼。
- 我補上的程式在做什麼：說明這列的螢幕時間有沒有超過 10 小時
- 執行或驗證遇到的問題與修正：無
- 檢查結果及我如何確認：透過上面的執行驗證程式所驗證。


## 📝 學生作業 02-Q2｜AI 查詢與作答：品質問題
**任務：**完整資料列重複與 student_id 重複有何不同？舉一個不能直接刪除的情境。

**可使用的 AI 提示詞：**
```text
我是 Python 與資料分析初學者。請舉例比較完整資料列重複與學生編號重複，並說明為何需先確認資料記錄意義。
請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
```
先閱讀教材，再詢問 AI。請記錄工具名稱、實際提示詞；若 AI 提供資料來源，記錄可查證的網址／文件名稱與日期，並自行檢查，無法查證時寫「尚未查證」。不要提供真實個資。


### ✍️ 作業 02-Q2 專用作答區：請雙擊此文字儲存格填寫
- AI 工具及實際提示詞：kklc；我是 Python 與資料分析初學者。請舉例比較完整資料列重複與學生編號重複，並說明為何需先確認資料記錄意義。請使用繁體中文、台灣常用名詞，分步解釋，並提出我可以自行驗證的一個例子；不要假設我的程式輸出。
- AI 回答重點（請自行整理）：
- | | **完整資料列重複** | **學生編號重複** |
|---|---|---|
| 比較什麼 | **整列所有欄位**都一模一樣 | 只看 `student_id` 這一欄有沒有出現兩次 |
| 通常代表 | 同一筆紀錄被重複貼上（真的多出來了） | 同個學生有多筆紀錄（**可能正常，也可能異常**） |
| 該不該刪 | 幾乎該刪（多出來的那份） | **要先確認「一列代表什麼」才能決定** |
關鍵差別：學生編號重複不一定是錯誤。 一學生考多場試，編號當然會出現多次——這時「整列重複」才是要刪的目標。但若資料設計是「一學生一列」，編號重複就是出錯。
- 我的理解與具體例子：
用一個 5 列的範例看兩種重複

假設資料如下（ 以下是我虛構的示範資料，不是你的 df）：

| 列索引 | student_id | quiz_number | screen_time_hours | quiz_score |
|---|---|---|---|---|
| 0 | A | 1 | 10 | 85 |
| 1 | B | 1 | 5 | 70 |
| 2 | A | 1 | 10 | 85 |
| 3 | A | 2 | 12 | 90 |
| 4 | C | 1 | 8 | 66 |

我的沙箱（pandas 3.0.2）跑出的結果：

完整列重複：只有 第 0、2 列——這兩列每個欄位都完全一樣，是真正多出來的一份。
學生編號重複： 第 0、2、3 列——因為 A 出現了三次。
注意第 3 列：它的 student_id 也是 A，但內容是「第 2 場考、12 小時、90 分」，跟第 0 列 完全不同。所以：

「完整列重複」抓出 2 列；「學生編號重複」抓出 3 列。 同一份資料，兩種「重複」抓出的範圍不一樣。 第 3 列就是兩者之間最要命的差異。

為什麼必須先確認「一列代表什麼」
因為「學生編號重複」是 對是錯，取決於資料的觀察單位：

| 情境 | 一列代表 | 學生編號重複 | 該怎麼處理 |
|---|---|---|---|
| A：每列 = 一次測驗成績 | 一筆「人 × 試卷」紀錄 | **正常**（一人考多場） | 只刪「完整列重複」；按 `student_id` 去重會**誤刪**第 2 場考 |
| B：每列 = 一個學生（寬表，一人一列） | 一個學生 | **錯誤**（同一人不該有兩列） | 該刪重複編號，但要先查是「貼重」還是「兩筆都要留」 |
| C：每列 = 一天 | 一筆「人 × 日期」紀錄 | **正常**（一人多日） | 兩欄 `student_id` + 日期都一樣才該刪 |

用 drop_duplicates(subset=["student_id"]) → 只留 A 的 第一筆、B、C。 A 的第 2 場考（第 3 列）被誤刪——若情境是 A（每列一次測驗），你就永久丟掉了合法資料。
用 drop_duplicates()（整列比對）→ 只刪第 2 列。若情境是 B（一人一列），則 A 的兩筆重複還是留著，錯誤沒被清掉。

- 查證來源／方式及日期（無法查證請註明）：

查證日期： 2026-11-11（台北時間，UTC+08:00）
查證方式： 直接抓取 pandas 官方 API 文件全文（文件版號 3.1.1）
來源： pandas.DataFrame.duplicated 
pandas.pydata.org、pandas.DataFrame.drop_duplicates、pandas.pydata.org

- 我檢查或修正 AI 回答的地方：無


## 2.3 日期格式與未知值：獨立銷售情境
**觀念說明：**`01-02-2026` 在日／月與月／日規則下可能是不同日期，不能猜。以下例子只轉換明確的 ISO 形式，以及日月數字相同、沒有日月歧義的 `01-01-2026`；「未知」、其他格式先保留為待查。

**提問：**`NaT` 是什麼？它是否表示 0 或可自行補成今天？

NaT 代表「(N)ot-(A)-(T)ime」，是日期時間資料的「缺失值」，相當於數值資料的 NaN。 官方文件定義：NaT 用來標記 pandas 中 datetime 與 timedelta 物件的缺失或 null 值，行為類似數值資料的 NaN； 包含 NaT 的運算通常會讓結果也變成 NaT
它是否表示 0？—— 否，絕對不能當 0
| 證據 | 說明 |
|---|---|
| `NaT == 0` → 不成立 | NaT 與 0 不相等 |
| `NaT > 0` → **直接拋 `TypeError`** | NaT 連跟數字比大小都不支援，因為它不是任何數字 |
| `NaT + 1 day` → NaT | 參與運算會「傳染」成 NaT，而不是算出結果 [1] |
| NaT 的內部 int64 值 = `-9223372036854775808`（int64 最小值） | 若硬把含 NaT 的日期欄位 `astype("int64")`，會得到這個**天文級負數**混進資料裡，之後平均、範圍會被拉爆 |

概念上：0 是「一個真實的時間點」（例如 1970-01-01 的 Unix epoch 0），NaT 是「 不知道是幾號」。把 NaT 當 0，等於把「不知道」說成「1970 年 1 月 1 日」——跟我們之前談的「缺失值 ≠ 數值零」是同一個陷阱，只是這次發生在日期上。

能自行補成今天嗎？—— 技術上可以，語義上通常是錯的

結果不可重現：同一份資料，今天跑和明年跑，補出的日期不同。
污染後續計算（實測，原始範圍只有 10 天）：

| 處理 | max − min |
|---|---|
| 不補，NaT 被跳過 | **10 天**（真實範圍）|
| 補成 `Timestamp(0)` | **20464 天**（多一個 1970 年）|
| 補成今天 | **314 天**（多一個 2026-11-11）|

一旦補上，「日期範圍」「相隔天數」「最晚一筆」全都會算錯，而且錯得不會報錯。

如果是「日期沒記錄到」→ 保留 NaT，統計時 pandas 會自動跳過，這是安全預設。
如果是「日期被弄壞」（拼錯、格式錯誤）→ 應回原始資料 修好，而不是補今天。
如果業務上真的需要「用某筆已知日期代表這筆」（例如「以最後一次紀錄的日期」）→ 用有根據的規則（如 ffill() 帶入前一筆）， 並在程式碼註解寫明這個假設。
要補成「今天」的唯一正當情境：欄位意義是「最後更新日」且系統確實今天更新過——這種情況原始資料就不該是 NaT 了。


In [5]:
# 四筆小資料專門用來示範日期品質，與學生資料主線分開
sales = pd.DataFrame({"sale_id": [1, 2, 3, 4],
                      "sale_date_raw": ["2026/01/01", "01-01-2026", "未知", "01-02-2026"],
                      "amount": [500, 800, 300, 600]})
sales_clean = sales.copy()  # 不改動原始資料
sales_clean["sale_date_clean"] = pd.NaT  # 新欄先設為無法確認
# 使用精確格式解析 YYYY/MM/DD；錯誤值標成 NaT，保留原值供追查
iso = sales_clean["sale_date_raw"].str.fullmatch(r"\d{4}/\d{2}/\d{2}")
sales_clean.loc[iso, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[iso, "sale_date_raw"], format="%Y/%m/%d", errors="coerce")
# 範例中的 01-01-2026 無日月歧義；只處理這個已確認值
same_day_month = sales_clean["sale_date_raw"].eq("01-01-2026")
sales_clean.loc[same_day_month, "sale_date_clean"] = pd.to_datetime(
    sales_clean.loc[same_day_month, "sale_date_raw"], format="%d-%m-%Y", errors="coerce")
# 任何尚未確認的資料都保留原值，列入人工查證清單
sales_clean["needs_review"] = sales_clean["sale_date_clean"].isna()
display(sales_clean)
print("待查筆數：", sales_clean["needs_review"].sum())
assert sales_clean.loc[3, "needs_review"]  # 有歧義日期沒有被任意轉換


,sale_id,sale_date_raw,amount,sale_date_clean,needs_review
0,1,2026/01/01,500,2026-01-01,False
1,2,01-01-2026,800,2026-01-01,False
2,3,未知,300,NaT,True
3,4,01-02-2026,600,NaT,True


待查筆數： 2


## 延伸示範：彙整品質檢查紀錄
缺失、重複與超出範圍是不同問題，處理前應先有可追查的摘要。以下範圍上限只適用模擬資料生成規格。

**驗證提問：**資料筆數是否變少？為何 `sleep_hours` 的缺失不能直接補 0？日期未能轉換時，應保留原文，並洽資料來源確認規則。

1.「資料筆數是否變少？」→ 你目前的流程：不會變少

關鍵區分：「顯示的列數」不等於「資料筆數」。 df.loc[flag] 是「挑出來看」，像拍照，原資料不動 [附註：duplicated 等回傳布林 Series 的方法亦同，見前輪文件。

2.「為何 sleep_hours 的缺失不能直接補 0？」→ 補 0 會把『沒回答』當成『沒睡覺』，且永遠無法挽回

| 處理 | 平均 |
|---|---|
| 保留 NaN（pandas 統計自動跳過 [2]） | **8.0** |
| `fillna(0)` 補零 | **6.0** ← 被拉低 |
| 對照組：`[7, 8, 0, 9]`（0 是真實測量） | **6.0** ← **和補 0 完全一樣** |

第三列這個「完全一樣」就是問題核心： 補 0 之後，「沒回答的人」和「真的 0 小時的人」變成同一個數字，資料裡再也分不出來（isna().sum() 從 1 變成 0，證據消失）

語義：0 是有效測量值，NaN 是無資料——這與我們第一輪的原則相同，套到睡眠欄位就是「漏填 ≠ 沒睡」。
本欄位的特殊性（實務判斷，非文件）：生理上「0 小時睡眠」極不可能，所以問卷資料裡的 0 反而是 可疑訊號——可能是哨兵值（填 0 代表沒填）。這更要先查資料字典，而不是拿 0 去補其他缺漏。


In [6]:
# 先彙整品質數量，不直接變更或刪除資料
quality_summary = pd.Series({
    "rows": len(df),
    "sleep_missing": int(df["sleep_hours"].isna().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "duplicate_ids": int(df["student_id"].duplicated().sum()),
    "screen_above_10": int((df["screen_time_hours"] > 10).sum())})
display(quality_summary)
assert quality_summary["rows"] == 60


rows               60
sleep_missing       2
duplicate_rows      0
duplicate_ids       0
screen_above_10     1
dtype: int64

## 本單元學習收束與 AI 使用紀錄

請學生先用自己的話解釋一個關鍵觀念，再指出一個執行結果及驗證方式。記錄：使用的提示詞、AI 回答中檢查或修改的地方、遇到的問題、修正後的證據與本資料限制。

回答：

在任務二中，我學到了篩選資料品質的部分，如缺失值該如何處理，就先分辨出此缺失值的意義，0就是真正數值上的0；而NaN式系統沒抓到或是根本沒資料；未知則是具達或是使用者回答模糊，接下來是重複、異常值及極端值的部分，最後就是日期的判斷與呈現。

這裡的執行結果會抓出異常狀況的欄位及數值，而指令皆是使用講義中提供的提示詞，一樣會使用AI給的例子做驗證，測試是否能使用，大多都不需修正，而這邊換了kklc做使用後，就能有查證的資料及日期了。

**通用提示詞**：我是 Python 初學者，請逐步協助我完成上面的任務；每段程式加繁體中文註解，說明輸入、處理與輸出，提供兩項可實際執行的驗證，保留原始資料，不要虛構執行結果。
